This is a short notebook to retrieve datasets from NFDI4Earth fulfilling specific conditions. The data retrieved can be stored as json afterwards.
The following metadata are currently considered (Corresponding query: *access_desired_attributes_only*)


*   Dataset title
*   Dataset description
*   Dataset url (a.ka. landing page)
*   Dataset download url
*   Temporal coverage (start date and end date)
*   Location





In [ ]:
!pip install sparqlwrapper

# Query datasets

## Specify endpoint and formulate queries

In [100]:
from SPARQLWrapper import SPARQLWrapper, JSON

endpoint = SPARQLWrapper("https://sparql.knowledgehub.nfdi4earth.de/")
endpoint.setReturnFormat(JSON)


access_available_attributes_for_datasets = """
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX dcat: <http://www.w3.org/ns/dcat#>

SELECT DISTINCT ?predicate
WHERE {
  ?entity rdf:type dcat:Dataset;
          ?predicate ?attributeInformation.
}

"""

access_datasets_and_all_attributes = """

PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX dcat: <http://www.w3.org/ns/dcat#>
PREFIX geo: <http://www.opengis.net/ont/geosparql#>
PREFIX schema: <http://schema.org/>
PREFIX dct: <http://purl.org/dc/terms/>
PREFIX locn: <http://www.w3.org/ns/locn#>

SELECT *
WHERE {
  ?dataset rdf:type dcat:Dataset;
             dct:title ?title;
             schema:description ?description;
             #dcat:landingPage ?landingpage;
             dct:creator ?bnode_creator;
             dcat:distribution ?bnode_distrib;
             dct:temporal ?bnode_temporal;
             dct:spatial ?bnode_spatial.

   ?bnode_creator schema:name ?autorname.
   ?bnode_distrib dcat:downloadURL ?download_url.

   ?bnode_temporal dcat:startDate ?startDate;
                 dcat:endDate ?endDate.
   ?bnode_spatial locn:geometry ?location.
}

"""


access_dataset_title = """

PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX dcat: <http://www.w3.org/ns/dcat#>
PREFIX geo: <http://www.opengis.net/ont/geosparql#>
PREFIX schema: <http://schema.org/>
PREFIX dct: <http://purl.org/dc/terms/>
PREFIX locn: <http://www.w3.org/ns/locn#>

SELECT DISTINCT ?title

WHERE {
  ?dataset rdf:type dcat:Dataset;
             dct:title ?title;
             schema:description ?description;
             #dcat:landingPage ?landingpage;
             dct:creator ?bnode_creator;
             dcat:distribution ?bnode_distrib;
             dct:temporal ?bnode_temporal;
             dct:spatial ?bnode_spatial.

   ?bnode_creator schema:name ?autorname.
   ?bnode_distrib dcat:downloadURL ?download_url.

   ?bnode_temporal dcat:startDate ?startDate;
                 dcat:endDate ?endDate.
   ?bnode_spatial locn:geometry ?location.
}

"""


access_desired_attributes_only = """

PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX dcat: <http://www.w3.org/ns/dcat#>
PREFIX geo: <http://www.opengis.net/ont/geosparql#>
PREFIX schema: <http://schema.org/>
PREFIX dct: <http://purl.org/dc/terms/>
PREFIX locn: <http://www.w3.org/ns/locn#>

SELECT DISTINCT ?title (GROUP_CONCAT(DISTINCT ?authorName; separator=", ") AS ?authors) ?description ?landingpage ?download_url ?startDate ?endDate ?location

WHERE {
  ?dataset rdf:type dcat:Dataset;
             dct:title ?title;
             schema:description ?description;
             dcat:landingPage ?landingpage;
             dct:creator ?bnode_creator;
             dcat:distribution ?bnode_distrib;
             dct:temporal ?bnode_temporal;
             dct:spatial ?bnode_spatial.

   ?bnode_creator schema:name ?autorname.
   ?bnode_distrib dcat:downloadURL ?download_url.

   ?bnode_temporal dcat:startDate ?startDate;
                 dcat:endDate ?endDate.
   ?bnode_spatial locn:geometry ?location.
}

"""

## Execute queries




In [101]:
import json
import time

datasets = []
n_iterations = 10 # virtuoso gives back 10000 rows max at once. If you want to retrieve more than that, you need to iterate, e.g. you need 2 iterations if you want to retrieve 18000 items, or 50 iterations for 500 000 items and so on.

try:

    for i in range(n_iterations):

      # pick the query to execute
      endpoint.setQuery(access_desired_attributes_only + " OFFSET " + str(i) + "0000" + " LIMIT 10000") # since virtuoso returns 10000 rows max, we pick 10000 max per iteration.

      response = endpoint.queryAndConvert()

      all_results = response["results"]["bindings"]

      print("Iteration: ",  str(i+1),  "; the number of results is ", len(all_results))

      for result in all_results:

        datasets.append(result)


    # format the results as json strings
    results_as_json_str = json.dumps(datasets)

    timestr = time.strftime("%Y%m%d-%H%M%S")
    json_path = "sample_" + timestr + ".json"

    # optional: save all results in a file
    with open(json_path, "w") as f:
      f.write(results_as_json_str)

except Exception as e:
    print(e)

Iteration:  1 ; the number of results is  10000
Iteration:  2 ; the number of results is  10000
Iteration:  3 ; the number of results is  10000
Iteration:  4 ; the number of results is  10000
Iteration:  5 ; the number of results is  10000
Iteration:  6 ; the number of results is  10000
Iteration:  7 ; the number of results is  8305
Iteration:  8 ; the number of results is  0
Iteration:  9 ; the number of results is  0
Iteration:  10 ; the number of results is  0


## Display results

In [102]:
print ("The number of results is: ", len(datasets))

The number of results is:  68305


Read the results and display the first 10

In [103]:
data = json.loads(results_as_json_str)

for i, datum in enumerate(data):
    if i < 10:
      print(datum)

{'title': {'type': 'literal', 'value': 'Soilsurface temperatures in 2 cm depth between summer 2018 and 2019 with iButton-sensor 046 in Illirney, Russia'}, 'authors': {'type': 'literal', 'value': ''}, 'description': {'type': 'literal', 'value': 'Soil surface temperatures were collected with help of iButtons along different transects and clusters in the region of Illirney (Russia). The interval was set to 4 hours resp. 4.25 hours. Different models of iButtons were used (with a resolution of 0.0625 °C and 0.5°C resp.), see Comment.'}, 'landingpage': {'type': 'typed-literal', 'datatype': 'http://www.w3.org/2001/XMLSchema#anyURI', 'value': 'https://doi.org/10.1594/PANGAEA.915201'}, 'download_url': {'type': 'typed-literal', 'datatype': 'http://www.w3.org/2001/XMLSchema#anyURI', 'value': 'https://doi.pangaea.de/10.1594/PANGAEA.915201?format=textfile'}, 'startDate': {'type': 'literal', 'value': '2018-07-09T00:00:00'}, 'endDate': {'type': 'typed-literal', 'datatype': 'http://purl.org/dc/terms/P

Store results in a file (takes time for large files, so commented by default)

In [104]:
'''
import json

with open(json_path, "r") as file:
    data = json.load(file)

print(data)
'''

'\nimport json\n\nwith open(json_path, "r") as file:\n    data = json.load(file)\n\nprint(data)\n'